<!-- nav -->
[← Organizing a design](03_Organizing_A_Design.ipynb) · [Home](../../README.md) · [System architecture and interaction models →](../../docs/SYSARCH.md)

# Checking and saving

Two kinds of check apply to an architecture. mbse-schemas' `Validators.Validate` checks what the schemas say:
scalar types, the relations' `unique` clauses (one module per instance, one name per interface), which schemas may fill
a link. `Interactions.Check` checks what the architecture means and no `unique` clause can say. Both list every
problem, each a line naming what it is about, and neither stops at the first.

Then a design is saved, as a store's snapshot, and loaded into another store.

In [1]:
from mbse.Schemas.Framework import Builders, JSON, Validators
from mbse.Architecture.Systems.Examples import Pipeline
from mbse.Architecture.Systems.Framework import Interactions

store = Builders.Store()
p = Pipeline.build(store)
print(Validators.Validate(store).Reachable(p.schemas.Module, p.pipeline), Interactions.Check()(p.pipeline))

[] []


## Breaking it on purpose

Three mistakes: a channel added without a message schema, attached only to a destination; a port given no
direction; and the logger made an instance of a second module.

In [2]:
with store.transaction():
    store.Interactions.Implementation(p.impl, lambda x: x.channels(lambda a: a.insert(lambda e: e.name("debug").channel(
        lambda c: c))))
    debug = p.impl.channels.at("debug").channel
    Pipeline.connect(store, debug, p.instance["logger"], "records", "rx")
    store.Interactions.Module(p.sensor, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("trigger").intf(
        lambda i: i.ports(lambda a: a.insert(lambda e: e.name("go").port(lambda q: q)))))))
    store.Interactions.Instance(p.instance["logger"], lambda i: i.module(lambda a: a.insert(lambda e: e.module(p.filter))))

print("Check:")
for problem in Interactions.Check()(p.pipeline):
    print(" ", problem)
print("Validate:")
for problem in Validators.Validate(store).Reachable(p.schemas.Module, p.pipeline):
    print(" ", problem)

Check:
  Interactions.Channel 'reference.debug' carries messages of no schema
  Interactions.Channel 'reference.debug' has no source
  Interactions.Port 'Sensor.trigger.go' has no direction: give it 'in' or 'out'
Validate:
  unique(module) violated: entries agreeing on ['instance'] differ on ['module']


Each line names the object by its schema and its path: `Interactions.Channel 'reference.debug'` is the channel
`debug` of the implementation `reference`. Fixing each is a write like any other.

In [3]:
with store.transaction():
    store.Interactions.Channel(debug, lambda c: c.message(lambda a: a.insert(lambda e: e.schema(p.Sample))))
    Pipeline.connect(store, debug, p.instance["filter"], "filtered", "tx")
    go = p.sensor.interfaces.at("trigger").intf.ports.at("go").port
    store.Interactions.Port(go, lambda q: q.direction("in"))
    store.Interactions.Instance(p.instance["logger"], lambda i: i.module(lambda a: a.remove(lambda e: e.module(p.filter))))
print(Interactions.Check()(p.pipeline), Validators.Validate(store).Reachable(p.schemas.Module, p.pipeline))

[] []


## Saving and loading

The library reaches every part of the design: its modules own their interfaces and ports, each module reaches its
implementations, and they reach their instances, channels and soft containers. So the store's snapshot holds the
whole design, with the schemas it uses, `dsp.Sample` included.

In [4]:
text = JSON.ToJSON(store).Snapshot()
print(len(text), "characters")

copy = Builders.Store()
JSON.FromJSON(copy).Snapshot(text)
Interactions.define(copy)
module = copy.singleton("Interactions.Library").modules.at("Pipeline").module
print(Interactions.Check()(module), JSON.ToJSON(copy).Snapshot() == text)

75452 characters
[] True


Call `Interactions.define` on the store you load into, before or after loading. The snapshot brings the schemas, but
not the extension `define` makes to the store's meta-schemas, which lets a schema object such as `dsp.Sample` fill
`Interactions.HasSchema`'s `schema` link. Without it, validation reports every channel's message schema.

A channel's message schema is written as the schema's path, `dsp.Sample`, so the loading store links its own
`dsp.Sample`.

In [5]:
channel = next(iter(module.implementations)).impl.channels.at("raw").channel
print(Interactions.message_schema(channel) is copy.registered("dsp.Sample"))

True


That is the whole of the package: modules and their interfaces, implementations of them, their organization, and how
to check and keep them. The [skill](https://github.com/pitaman71/mbse-sysarch/blob/main/skills/mbse-sysarch/SKILL.md)
condenses it for reference, and [the design](../../docs/SYSARCH.md) records every decision and what is still open.

<!-- nav -->
[← Organizing a design](03_Organizing_A_Design.ipynb) · [Home](../../README.md) · [System architecture and interaction models →](../../docs/SYSARCH.md)